# 七天终点路径复核：只读核对
在本证据包目录运行。此 notebook 仅读取已保存结果，不重新分类、不训练、不改原数据。每条记录对应一个第七天终点；同一 drifter 可跨类别。

In [1]:
from pathlib import Path
import pandas as pd
p = Path.cwd()
s = pd.read_csv(p/'results/updated_summary.csv')
print(s[s.scope.eq('frozen_used')][['cohort','state','pairs','ids','pair_fraction']].to_string(index=False))

    cohort              state  pairs  ids  pair_fraction
     train grounding_evidence      0    0       0.000000
     train  drifting_evidence 245191  332       0.923879
     train          uncertain  20202  249       0.076121
     train   endpoint_missing      0    0       0.000000
validation grounding_evidence      0    0       0.000000
validation  drifting_evidence  30949   45       0.911042
validation          uncertain   3022   38       0.088958
validation   endpoint_missing      0    0       0.000000


In [2]:
d = pd.read_csv(p/'results/uncertain_breakdown.csv')
print(d[d.scope.eq('frozen_used')].to_string(index=False))

      scope     cohort               path_evidence  pairs  ids  original_uncertain_pairs
frozen_used      train   consistent_drift_evidence    278   53                     22279
frozen_used      train            looping_evidence   1799  114                     22279
frozen_used      train        suspected_local_stay    258    2                     22279
frozen_used      train insufficient_or_conflicting   3647   67                     22279
frozen_used      train            other_unresolved  16297  230                     22279
frozen_used validation   consistent_drift_evidence     20    6                      3476
frozen_used validation            looping_evidence    434   18                      3476
frozen_used validation        suspected_local_stay    371    1                      3476
frozen_used validation insufficient_or_conflicting    177    8                      3476
frozen_used validation            other_unresolved   2474   37                      3476


In [3]:
f = pd.read_parquet(p/'results/updated_endpoint_states.parquet')
assert not f.duplicated(['id','origin_source_obs','horizon_days']).any()
for cohort, total in [('train',265393),('validation',33971)]:
    g = f[f.used_by_original & f.cohort.eq(cohort)]
    assert len(g) == total
    counts = g.updated_state.value_counts()
    assert counts.sum() == total
    print(cohort, 'total', total, 'uncertain', counts.get('uncertain',0), 'uncertain IDs', g[g.updated_state.eq('uncertain')].id.nunique())
print('Keys unique; actual-used pair totals conserved. ID categories may overlap.')

train total 265393 uncertain 20202 uncertain IDs 249
validation total 33971 uncertain 3022 uncertain IDs 38
Keys unique; actual-used pair totals conserved. ID categories may overlap.
